# CFPB complaint classifier: Colab training
Runs `cfpb_mlops.training.train` from the GitHub repo against `complaints.csv` in your Google Drive.
Set the runtime to **High-RAM** if you can (Runtime > Change runtime type). Run cells top to bottom.
Output: `run_outputs.zip` (Drive + download) holding the model, per-class report, confusion matrix, class counts, `mlflow.db` and `run_info.json`.

In [ ]:
# 1. Settings
DRIVE_CSV = "/content/drive/MyDrive/cfpb/complaints.csv"  # where you uploaded the file
SAMPLE_N = 500_000   # most recent N rows with a narrative; None = all ~1.9M (needs lots of RAM)
MAX_FEATURES = 10_000
C = 1.0
TEST_MONTHS = 6
REGISTER = False

In [ ]:
# 2. Mount Drive, clone repo, copy the CSV onto local disk (much faster reads than Drive)
from google.colab import drive
drive.mount("/content/drive")

!rm -rf /content/CFPB-Complaint-Routing
!git clone -q https://github.com/Reehan09Sarmah/CFPB-Complaint-Routing.git /content/CFPB-Complaint-Routing
!mkdir -p /content/CFPB-Complaint-Routing/data
!cp "$DRIVE_CSV" /content/CFPB-Complaint-Routing/data/complaints.csv
!ls -lh /content/CFPB-Complaint-Routing/data
!git -C /content/CFPB-Complaint-Routing log -1 --oneline
!free -h | head -2
!pip -q install mlflow tqdm

In [ ]:
# 3. Train (calls train() directly because the CLI cannot pass sample_n=None)
import os, sys
sys.path.insert(0, "/content/CFPB-Complaint-Routing/src")

import mlflow
from cfpb_mlops.training import train as T

assert hasattr(T, "diagnostics"), "Old code cloned: push the latest train.py to GitHub first"

os.chdir("/content")  # sqlite store + ./mlruns artifacts land here
mlflow.set_tracking_uri("sqlite:////content/mlflow.db")
T.train(
    sample_n=SAMPLE_N,
    max_features=MAX_FEATURES,
    C=C,
    test_months=TEST_MONTHS,
    register_model=REGISTER,
)

In [ ]:
# 4. Collect everything into ONE folder, verify it, zip it, save to Drive, download
import glob, json, os, pathlib, shutil, subprocess
import mlflow, sklearn

ROOT = pathlib.Path("/content")
REPO = ROOT / "CFPB-Complaint-Routing"

runs = mlflow.search_runs(
    experiment_names=["cfpb-complaint-classifier"], order_by=["start_time DESC"]
)
run = runs.iloc[0]
run_id = run["run_id"]

out = ROOT / "run_outputs"
shutil.rmtree(out, ignore_errors=True)
out.mkdir()

mlflow.artifacts.download_artifacts(run_id=run_id, dst_path=str(out))  # diagnostics/
model_dirs = sorted(glob.glob(str(ROOT / "mlruns/*/models/*/artifacts")), key=os.path.getmtime)
shutil.copytree(model_dirs[-1], out / "model")  # newest logged model
shutil.copy(ROOT / "mlflow.db", out / "mlflow.db")

commit = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True
).stdout.strip()
(out / "run_info.json").write_text(
    json.dumps(
        {
            "run_id": run_id,
            "git_commit": commit,
            "sklearn": sklearn.__version__,
            "mlflow": mlflow.__version__,
            "settings": dict(
                SAMPLE_N=SAMPLE_N, MAX_FEATURES=MAX_FEATURES, C=C, TEST_MONTHS=TEST_MONTHS
            ),
            "metrics": {k[8:]: v for k, v in run.items() if k.startswith("metrics.")},
            "params": {k[7:]: v for k, v in run.items() if k.startswith("params.")},
        },
        indent=2,
        default=str,
    )
)

# Verify before we trust it
need = [
    "diagnostics/classification_report.json",
    "diagnostics/confusion_matrix.csv",
    "diagnostics/class_counts.json",
    "model/MLmodel",
    "run_info.json",
    "mlflow.db",
]
missing = [f for f in need if not (out / f).exists()]
assert not missing, f"Missing outputs: {missing}"
assert list((out / "model").glob("model.*")), "No model file in model/"
print(json.dumps(json.loads((out / "run_info.json").read_text())["metrics"], indent=2))

shutil.make_archive(str(ROOT / "run_outputs"), "zip", out)
!ls -lh /content/run_outputs.zip
!cp /content/run_outputs.zip /content/drive/MyDrive/cfpb/run_outputs.zip
from google.colab import files
files.download("/content/run_outputs.zip")